In [ ]:
# =====================================================
# preprocessNoiseAudio_STD1-96kHz
# Hardware: STD1-028K + Voltage mode + Focusrite 2i2
# =====================================================

import os
import numpy as np
import librosa
import soundfile as sf
from scipy.signal import butter, filtfilt

# =========================
# 1. 路径设置（自动创建）
# =========================

PA_DIR = os.path.expanduser("~/Desktop/PA")
NOISE_WAV_DIR = os.path.join(PA_DIR, "Noise_Recordings")
OUTPUT_DIR = os.path.join(PA_DIR, "Noise_Sample")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Noise input dir :", NOISE_WAV_DIR)
print("Noise output dir:", OUTPUT_DIR)

# =========================
# 2. 硬件匹配参数
# =========================

SR = 96000              # Focusrite 2i2
LOWCUT = 2000           # Hz（去结构噪声）
HIGHCUT = 40000         # Hz（STD1 click / 高频噪声）
FILTER_ORDER = 4

CHUNK_MS = 20           # 每个噪声样本 20 ms
CHUNK_LEN = int(CHUNK_MS / 1000 * SR)

# =========================
# 3. 带通滤波函数
# =========================

def bandpass_filter(x, sr, lowcut, highcut, order=4):
    nyq = 0.5 * sr
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype="band")
    return filtfilt(b, a, x)

# =========================
# 4. 处理所有噪声 WAV
# =========================

noise_files = [
    f for f in os.listdir(NOISE_WAV_DIR)
    if f.lower().endswith(".wav")
]

assert len(noise_files) > 0, "❌ NOISE_WAV_DIR 中没有 wav 文件"

counter = 0

for wav_name in noise_files:
    wav_path = os.path.join(NOISE_WAV_DIR, wav_name)
    x, sr = librosa.load(wav_path, sr=SR, mono=True)

    x = bandpass_filter(x, sr, LOWCUT, HIGHCUT)

    num_chunks = len(x) // CHUNK_LEN

    for i in range(num_chunks):
        chunk = x[i*CHUNK_LEN:(i+1)*CHUNK_LEN]

        # 能量太小的直接跳过（避免静音）
        if np.max(np.abs(chunk)) < 1e-4:
            continue

        chunk = chunk / np.max(np.abs(chunk))  # 归一化

        out_name = f"noise_{counter:05d}.wav"
        out_path = os.path.join(OUTPUT_DIR, out_name)
        sf.write(out_path, chunk, sr)

        counter += 1

print("====================================")
print(f"Noise samples generated: {counter}")
print("Saved to:", OUTPUT_DIR)
print("====================================")
